# Geospatial Trail Data Exploration

This notebook loads processed trail geometry data, validates the spatial features, projects them into a Michigan-friendly coordinate reference system, and computes a geospatial trail length in miles.

The goal is to compare reported trail lengths with lengths derived from the stored geometries and then inspect the results using interactive mapping.

# Data loading and environment setup

Import required geospatial libraries and define paths for processed trail data.

This section also sets the coordinate reference system for Michigan and a conversion factor between meters and miles.

In [18]:
from pathlib import Path

import geopandas as gpd

In [19]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_PATH = PROCESSED_DIR / "dnr_up_hiking_trails_grouped.parquet"

MICHIGAN_GEOREF = "EPSG:3078"
METERS_PER_MILE = 1609.344

trails = gpd.read_parquet(PROCESSED_PATH)

trails = gpd.GeoDataFrame(trails).copy()

epsg_original = trails.crs.to_epsg()

print(f"Source CRS: EPSG:{epsg_original}")

Source CRS: EPSG:4326


# Geometry validation

Inspect the loaded trail geometries for missing, empty, or invalid shapes. This helps catch data quality issues before computing lengths or reprojecting the dataset.

In [ ]:
print(f"Missing geometries: {trails.geometry.isna().sum()}")
print(f"Empty geometries: {trails.geometry.is_empty.sum()}")
print(f"Invalid geometries: {(~trails.geometry.is_valid).sum()}")

trails.geometry.geom_type.value_counts()

Missing geomtries: 0
Empty geometries: 0
Invalid geomtries: 0


MultiLineString    107
LineString          31
Name: count, dtype: int64

# Reproject data for accurate distance calculations

Convert trail geometries into a projected CRS appropriate for Michigan so length measurements are computed in meters before converting to miles.

In [21]:
projected_trails = trails.to_crs(MICHIGAN_GEOREF)

epsg_projected = projected_trails.crs.to_epsg()

print(f"Projected CRS: EPSG:{epsg_projected}")

Projected CRS: EPSG:3078


# Compute geospatial trail length

Calculate a new length field from the projected geometry and convert from meters to miles for direct comparison with reported values.

In [22]:
projected_trails["GeospacialLength"] = (projected_trails.geometry.length / METERS_PER_MILE)

# Preview the projected trail dataset

Display the first few rows of the reprojected GeoDataFrame to verify the new geometry and computed length fields.

In [23]:
projected_trails.head()

,TrailGroupName,LengthCategory,HikingName,County,ReportedLengthMiles,TrailWidth,SurfaceTypes,AccessibilityValues,TrailStatuses,FacilityName,SegmentCount,geometry,GeospacialLength
0,Alger | Fox River Pathway,Long,Fox River Pathway,Alger,14.345076,0 To 2 Feet,Dirt Natural,Not Accessible,Open,State Forest,11,"MULTILINESTRING ((482473.921 670693.795, 48247...",14.345078
1,Alger | Laughing Whitefish Falls Trails,Medium,Laughing Whitefish Falls Trails,Alger,2.575298,0 To 2 Feet,Dirt Natural,Not Accessible,Open,SPRK422,17,"MULTILINESTRING ((417608.354 648542.593, 41760...",2.575299
2,Alger | North Country Trail,Long,North Country Trail,Alger,96.990313,Varies,"Concrete, Dirt Natural",Not Accessible,Open,SPRK422,129,"MULTILINESTRING ((504858.29 678975.86, 504874....",96.990289
3,Alger | Tyoga Historical Pathway,Short,Tyoga Historical Pathway,Alger,1.252893,0 To 2 Feet,Dirt Natural,Not Accessible,Open,State Forest,2,"MULTILINESTRING ((421071.54 662903.198, 421087...",1.252894
4,Alger | Wagner Falls Scenic Site - Foot Trail,Short,Wagner Falls Scenic Site - Foot Trail,Alger,0.125517,0 To 2 Feet,Dirt Natural,Not Accessible,Open,Wagner Falls Scenic Site,1,"LINESTRING (449931.179 648754.594, 449931.359 ...",0.125517


# Compare reported and measured lengths

Check whether the reported trail lengths differ from the computed geospatial lengths after rounding to three decimal places.

In [24]:
(projected_trails["ReportedLengthMiles"].round(3) !=  projected_trails["GeospacialLength"].round(3)).sum()

np.int64(2)

# Summarize length differences

Create an absolute difference field and summarize the distribution of discrepancies between reported and geospatially measured lengths.

In [25]:
projected_trails["LengthDifferenceMiles"] = (
    projected_trails["ReportedLengthMiles"]
    - projected_trails["GeospacialLength"]
).abs()

projected_trails["LengthDifferenceMiles"].describe().round(7)

count    1.380000e+02
mean     1.600000e-05
std      1.280000e-04
min      0.000000e+00
25%      1.000000e-07
50%      3.000000e-07
75%      8.000000e-07
max      1.435600e-03
Name: LengthDifferenceMiles, dtype: float64

# Inspect records where lengths differ

List any trails whose reported lengths and geospatially measured lengths do not match after rounding. These are candidates for manual review or data correction.

In [26]:
projected_trails[(projected_trails["ReportedLengthMiles"].round(3) !=  projected_trails["GeospacialLength"].round(3))]

,TrailGroupName,LengthCategory,HikingName,County,ReportedLengthMiles,TrailWidth,SurfaceTypes,AccessibilityValues,TrailStatuses,FacilityName,SegmentCount,geometry,GeospacialLength,LengthDifferenceMiles
25,Delta | Days River Pathway,Long,Days River Pathway,Delta,11.590940,0 To 2 Feet,Dirt Natural,Not Accessible,Open,State Forest,22,"MULTILINESTRING ((418743.326 593889.087, 41875...",11.590493,0.000447
47,Houghton | North Country Trail,Long,North Country Trail,Houghton,20.299796,Varies,Dirt Natural,Not Accessible,Open,Unknown,14,"MULTILINESTRING ((280265.482 681391.932, 28026...",20.298360,0.001436


# Visualize trails on a map

Render the trail dataset on an interactive map to visually inspect geometry placement and support spatial review.

In [27]:
trails.explore()